# Maximum-Sharpe PortfolioThis notebook studies the tangency portfolio, its capital-market-line interpretation, and the sensitivity of optimal weights to estimated returns.**Data:** Ken French 30 industry portfolios. **Estimation window:** January 1996–December 2000. All optimisation is long-only and fully invested.

In [ ]:
%matplotlib inlineimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport risk_kit as rkreturns = rk.get_ind_returns().loc['1996':'2000']er = rk.annualize_rets(returns, 12)cov = returns.cov() * 12risk_free_rate = 0.03print(f'Assets: {len(er)} | observations: {len(returns)}')

## 1. Optimise the tangency portfolioThe maximum-Sharpe portfolio maximises excess expected return per unit of volatility, subject to the long-only and full-investment constraints.

In [ ]:
weights = rk.msr(risk_free_rate, er, cov)msr_weights = pd.Series(weights, index=er.index, name='MSR weight').sort_values(ascending=False)msr_weights.to_frame().head(10)

In [ ]:
msr_return = rk.portfolio_return(weights, er)msr_volatility = rk.portfolio_vol(weights, cov)msr_sharpe = (msr_return - risk_free_rate) / msr_volatilityprint(f'Expected return: {msr_return:.2%}')print(f'Volatility:      {msr_volatility:.2%}')print(f'Sharpe ratio:    {msr_sharpe:.3f}')print(f'Weight sum:      {weights.sum():.8f}')

## 2. Capital market lineThe capital market line joins the risk-free asset to the tangency portfolio. It represents theoretical combinations of lending/borrowing at the risk-free rate and holding the tangency portfolio; borrowing constraints and real-world frictions are not included here.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))rk.plot_ef(80, er, cov, riskfree_rate=risk_free_rate, show_cml=True, show_ew=True, show_gmv=True, ax=ax)ax.set_title('30-industry frontier and maximum-Sharpe portfolio')plt.tight_layout()

## 3. Expected-return estimation sensitivityThe covariance matrix is held fixed while expected returns are changed. This isolates how sensitive the optimiser is to the return estimates supplied to it.

In [ ]:
assets = ['Food', 'Fin']base_er = er.loc[assets].to_numpy()asset_cov = cov.loc[assets, assets]scenarios = {    'Historical estimates': base_er,    'Food +1 pp': base_er + np.array([0.01, 0.00]),    'Fin +1 pp': base_er + np.array([0.00, 0.01]),    'Both +1 pp': base_er + np.array([0.01, 0.01]),}scenario_weights = pd.DataFrame({k: rk.msr(risk_free_rate, v, asset_cov) for k, v in scenarios.items()}, index=assets)scenario_weights

## 4. Interpretation and limitations- The tangency portfolio depends on both expected returns and the covariance matrix.- Maximum-Sharpe weights can be highly sensitive to small expected-return changes.- The risk-free rate is an illustrative annual assumption, not a live market quote.- The analysis is in-sample, long-only, and excludes costs, taxes, liquidity constraints and out-of-sample execution.